In [1]:
"""
=============================================================================
NLP Architectures: RNN, LSTM, BERT, RoBERTa
=============================================================================
This file provides from-scratch implementations (RNN, LSTM) using PyTorch,
and fine-tuning / inference wrappers for pretrained BERT and RoBERTa via
HuggingFace Transformers. Each section ends with a self-contained test.

Dependencies:
    pip install torch transformers datasets

Run:
    python nlp_architectures.py
=============================================================================
"""

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# ─────────────────────────────────────────────────────────────────────────────
# SECTION 1 — VANILLA RNN
# ─────────────────────────────────────────────────────────────────────────────
# A Recurrent Neural Network processes sequences step-by-step.
# At each time step t it receives:
#   - x_t  : the current input vector
#   - h_t-1: the hidden state from the previous step
# and produces:
#   - h_t = tanh(W_hh * h_t-1 + W_xh * x_t + b)
# The final hidden state is used for classification / regression.
#
# Limitation: suffers from vanishing gradients over long sequences.
# ─────────────────────────────────────────────────────────────────────────────

class VanillaRNN(nn.Module):
    """
    A simple RNN for sequence classification.

    Architecture
    ────────────
    Embedding → RNN (n_layers) → last hidden state → Linear → logits

    Parameters
    ──────────
    vocab_size  : int  — number of unique tokens
    embed_dim   : int  — embedding dimension
    hidden_dim  : int  — RNN hidden state dimension
    output_dim  : int  — number of target classes
    n_layers    : int  — number of stacked RNN layers
    dropout     : float — dropout probability between layers
    """

    def __init__(
        self,
        vocab_size: int,
        embed_dim: int,
        hidden_dim: int,
        output_dim: int,
        n_layers: int = 1,
        dropout: float = 0.3,
    ):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        # batch_first=True → input/output tensors shaped (batch, seq, feature)
        self.rnn = nn.RNN(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=n_layers,
            batch_first=True,
            dropout=dropout if n_layers > 1 else 0,
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, token_ids: torch.Tensor) -> torch.Tensor:
        """
        token_ids : (batch, seq_len)  — integer token indices
        returns   : (batch, output_dim) — class logits
        """
        # Step 1 — convert token IDs to dense embeddings
        embedded = self.dropout(self.embedding(token_ids))   # (B, T, E)

        # Step 2 — run through RNN; we only need the final hidden state
        _, hidden = self.rnn(embedded)                       # hidden: (layers, B, H)

        # Step 3 — take the top layer's hidden state and classify
        last_hidden = self.dropout(hidden[-1])               # (B, H)
        return self.fc(last_hidden)                          # (B, output_dim)


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 2 — LSTM (Long Short-Term Memory)
# ─────────────────────────────────────────────────────────────────────────────
# LSTM extends the RNN with a gating mechanism that controls information flow:
#
#   Forget gate : f_t = σ(W_f · [h_t-1, x_t] + b_f)
#   Input gate  : i_t = σ(W_i · [h_t-1, x_t] + b_i)
#   Cell update : g_t = tanh(W_g · [h_t-1, x_t] + b_g)
#   Cell state  : c_t = f_t ⊙ c_t-1 + i_t ⊙ g_t
#   Output gate : o_t = σ(W_o · [h_t-1, x_t] + b_o)
#   Hidden state: h_t = o_t ⊙ tanh(c_t)
#
# The cell state c_t acts as long-term memory; gating prevents vanishing gradients.
# ─────────────────────────────────────────────────────────────────────────────

class BidirectionalLSTM(nn.Module):
    """
    Bidirectional LSTM for sequence classification.

    Processes the sequence forwards AND backwards and concatenates
    both final hidden states before classification.

    Parameters
    ──────────
    vocab_size  : int
    embed_dim   : int
    hidden_dim  : int   — per direction; total = 2 * hidden_dim
    output_dim  : int
    n_layers    : int
    dropout     : float
    """

    def __init__(
        self,
        vocab_size: int,
        embed_dim: int,
        hidden_dim: int,
        output_dim: int,
        n_layers: int = 2,
        dropout: float = 0.3,
    ):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=n_layers,
            batch_first=True,
            bidirectional=True,                          # ← key difference
            dropout=dropout if n_layers > 1 else 0,
        )
        self.dropout = nn.Dropout(dropout)
        # Bidirectional doubles the hidden dimension
        self.fc = nn.Linear(hidden_dim * 2, output_dim)

    def forward(self, token_ids: torch.Tensor) -> torch.Tensor:
        """
        token_ids : (batch, seq_len)
        returns   : (batch, output_dim)
        """
        embedded = self.dropout(self.embedding(token_ids))  # (B, T, E)

        # _, (hidden, cell) — we ignore the per-step output and cell state
        _, (hidden, _) = self.lstm(embedded)
        # hidden shape: (n_layers * 2, B, H)  — 2 for bidirectional

        # Concatenate the top forward and backward hidden states
        # hidden[-2] = top forward layer, hidden[-1] = top backward layer
        top_hidden = torch.cat([hidden[-2], hidden[-1]], dim=1)  # (B, 2H)

        return self.fc(self.dropout(top_hidden))                 # (B, output_dim)


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 3 — BERT (Bidirectional Encoder Representations from Transformers)
# ─────────────────────────────────────────────────────────────────────────────
# BERT uses the Transformer encoder with two pre-training objectives:
#   1. Masked Language Modelling (MLM) — predict [MASK] tokens
#   2. Next Sentence Prediction  (NSP) — predict if two sentences are adjacent
#
# For downstream tasks we add a classification head on top of the [CLS] token
# and fine-tune the whole model end-to-end.
#
# Key features:
#   • Bidirectional — attends to both left and right context simultaneously
#   • Positional embeddings — learns absolute positions
#   • WordPiece tokenization — subword tokenization reduces OOV
# ─────────────────────────────────────────────────────────────────────────────

class BERTClassifier(nn.Module):
    """
    Fine-tunable BERT model for sequence classification.

    Loads pretrained weights from HuggingFace, adds a dropout + linear head
    on top of the pooled [CLS] token representation.

    Parameters
    ──────────
    model_name  : str  — HuggingFace model identifier
    output_dim  : int  — number of classes
    dropout     : float
    freeze_bert : bool — if True, only the head is trained (fast but weaker)
    """

    def __init__(
        self,
        model_name: str = "bert-base-uncased",
        output_dim: int = 2,
        dropout: float = 0.3,
        freeze_bert: bool = False,
    ):
        super().__init__()
        from transformers import BertModel
        self.bert = BertModel.from_pretrained(model_name)

        # Optionally freeze BERT parameters (useful for quick prototyping)
        if freeze_bert:
            for param in self.bert.parameters():
                param.requires_grad = False

        self.dropout = nn.Dropout(dropout)
        # bert-base hidden size = 768
        self.classifier = nn.Linear(self.bert.config.hidden_size, output_dim)

    def forward(
        self,
        input_ids: torch.Tensor,
        attention_mask: torch.Tensor,
        token_type_ids: torch.Tensor | None = None,
    ) -> torch.Tensor:
        """
        input_ids      : (batch, seq_len) — token IDs from BertTokenizer
        attention_mask : (batch, seq_len) — 1 for real tokens, 0 for padding
        token_type_ids : (batch, seq_len) — segment IDs (0=sent A, 1=sent B)

        returns        : (batch, output_dim) — logits
        """
        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
        )
        # pooler_output is the [CLS] representation processed through
        # a learned linear + tanh layer — BERT's default sentence embedding
        pooled = outputs.pooler_output            # (B, 768)
        return self.classifier(self.dropout(pooled))


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 4 — RoBERTa (Robustly Optimised BERT Pretraining Approach)
# ─────────────────────────────────────────────────────────────────────────────
# RoBERTa reuses BERT's architecture but improves training:
#   • No NSP task — removed as it hurt downstream performance
#   • Dynamic masking  — mask pattern changes each epoch, not fixed
#   • Larger batches + more data + longer training
#   • BPE tokenization (GPT-2 style) instead of WordPiece
#
# Result: consistently outperforms BERT on GLUE/SuperGLUE with same size.
# ─────────────────────────────────────────────────────────────────────────────

class RoBERTaClassifier(nn.Module):
    """
    Fine-tunable RoBERTa model for sequence classification.

    Identical classification head strategy to BERTClassifier; the main
    difference is that RoBERTa does NOT use token_type_ids.

    Parameters
    ──────────
    model_name  : str  — e.g. "roberta-base" or "roberta-large"
    output_dim  : int
    dropout     : float
    freeze_base : bool
    """

    def __init__(
        self,
        model_name: str = "roberta-base",
        output_dim: int = 2,
        dropout: float = 0.1,
        freeze_base: bool = False,
    ):
        super().__init__()
        from transformers import RobertaModel
        self.roberta = RobertaModel.from_pretrained(model_name)

        if freeze_base:
            for param in self.roberta.parameters():
                param.requires_grad = False

        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(self.roberta.config.hidden_size, output_dim)

    def forward(
        self,
        input_ids: torch.Tensor,
        attention_mask: torch.Tensor,
    ) -> torch.Tensor:
        """
        input_ids      : (batch, seq_len) — from RobertaTokenizer
        attention_mask : (batch, seq_len)

        Note: RoBERTa does NOT use token_type_ids.

        returns : (batch, output_dim) — logits
        """
        outputs = self.roberta(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )
        pooled = outputs.pooler_output
        return self.classifier(self.dropout(pooled))


# ─────────────────────────────────────────────────────────────────────────────
# SHARED UTILITIES
# ─────────────────────────────────────────────────────────────────────────────

def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    optimizer: optim.Optimizer,
    criterion: nn.Module,
    device: torch.device,
) -> float:
    """Run one full training epoch and return average loss."""
    model.train()
    total_loss = 0.0
    for batch in loader:
        optimizer.zero_grad()
        # Batch can be (input_ids, labels) or (input_ids, mask, labels)
        *inputs, labels = [t.to(device) for t in batch]
        logits = model(*inputs)
        loss = criterion(logits, labels)
        loss.backward()
        # Gradient clipping prevents exploding gradients (especially in RNNs)
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)


@torch.no_grad()
def evaluate(
    model: nn.Module,
    loader: DataLoader,
    device: torch.device,
) -> float:
    """Evaluate accuracy on a DataLoader."""
    model.eval()
    correct = total = 0
    for batch in loader:
        *inputs, labels = [t.to(device) for t in batch]
        logits = model(*inputs)
        preds = logits.argmax(dim=-1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
    return correct / total


# ─────────────────────────────────────────────────────────────────────────────
# SYNTHETIC DATA HELPERS
# ─────────────────────────────────────────────────────────────────────────────

def make_synthetic_rnn_data(
    n_samples: int = 200,
    seq_len: int = 20,
    vocab_size: int = 100,
    n_classes: int = 2,
    batch_size: int = 32,
):
    """
    Generate random integer sequences + random labels for RNN/LSTM smoke tests.
    Tokens are in [1, vocab_size-1] (0 reserved for padding).
    """
    X = torch.randint(1, vocab_size, (n_samples, seq_len))
    y = torch.randint(0, n_classes, (n_samples,))
    dataset = TensorDataset(X, y)
    return DataLoader(dataset, batch_size=batch_size, shuffle=True)


def make_synthetic_transformer_data(
    n_samples: int = 50,
    seq_len: int = 32,
    vocab_size: int = 30522,   # bert-base-uncased vocab size
    n_classes: int = 2,
    batch_size: int = 8,
    include_token_type: bool = True,
):
    """
    Generate random tensors that mimic tokenizer output for BERT / RoBERTa tests.
    Uses small seq_len and n_samples to stay fast without a GPU.
    """
    input_ids      = torch.randint(1, vocab_size, (n_samples, seq_len))
    attention_mask = torch.ones(n_samples, seq_len, dtype=torch.long)
    labels         = torch.randint(0, n_classes, (n_samples,))

    if include_token_type:
        token_type_ids = torch.zeros(n_samples, seq_len, dtype=torch.long)
        dataset = TensorDataset(input_ids, attention_mask, token_type_ids, labels)
    else:
        dataset = TensorDataset(input_ids, attention_mask, labels)

    return DataLoader(dataset, batch_size=batch_size, shuffle=True)


# ─────────────────────────────────────────────────────────────────────────────
# TEST SUITE
# ─────────────────────────────────────────────────────────────────────────────

def test_rnn():
    """
    Test 1: VanillaRNN — forward pass shape + 3 training steps.
    Expected: logits shape (B, 2), loss decreases across steps.
    """
    print("\n" + "=" * 60)
    print("TEST 1 — Vanilla RNN")
    print("=" * 60)

    VOCAB   = 100
    EMBED   = 32
    HIDDEN  = 64
    CLASSES = 2
    BATCH   = 16
    SEQ_LEN = 20

    device = torch.device("cpu")
    model  = VanillaRNN(VOCAB, EMBED, HIDDEN, CLASSES, n_layers=2, dropout=0.3).to(device)

    # ── Forward pass shape check ──────────────────────────────────────────────
    dummy_input = torch.randint(1, VOCAB, (BATCH, SEQ_LEN))
    logits = model(dummy_input)
    assert logits.shape == (BATCH, CLASSES), f"Bad shape: {logits.shape}"
    print(f"  [PASS] Forward pass → logits shape {tuple(logits.shape)}")

    # ── Mini training loop ────────────────────────────────────────────────────
    loader    = make_synthetic_rnn_data(n_samples=200, seq_len=SEQ_LEN,
                                        vocab_size=VOCAB, n_classes=CLASSES)
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()

    losses = []
    for epoch in range(3):
        loss = train_one_epoch(model, loader, optimizer, criterion, device)
        losses.append(loss)
        print(f"  Epoch {epoch+1}/3  loss={loss:.4f}")

    # ── Accuracy on same data (sanity — not a real eval) ─────────────────────
    acc = evaluate(model, loader, device)
    print(f"  Train acc (sanity): {acc:.2%}")
    print("  [PASS] VanillaRNN test complete")


def test_lstm():
    """
    Test 2: BidirectionalLSTM — forward pass shape + 3 training steps.
    Expected: logits shape (B, 2), runs without error.
    """
    print("\n" + "=" * 60)
    print("TEST 2 — Bidirectional LSTM")
    print("=" * 60)

    VOCAB   = 200
    EMBED   = 64
    HIDDEN  = 128
    CLASSES = 3   # multi-class this time
    BATCH   = 16
    SEQ_LEN = 30

    device = torch.device("cpu")
    model  = BidirectionalLSTM(VOCAB, EMBED, HIDDEN, CLASSES, n_layers=2, dropout=0.3).to(device)

    dummy_input = torch.randint(1, VOCAB, (BATCH, SEQ_LEN))
    logits = model(dummy_input)
    assert logits.shape == (BATCH, CLASSES), f"Bad shape: {logits.shape}"
    print(f"  [PASS] Forward pass → logits shape {tuple(logits.shape)}")

    loader    = make_synthetic_rnn_data(n_samples=300, seq_len=SEQ_LEN,
                                        vocab_size=VOCAB, n_classes=CLASSES)
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(3):
        loss = train_one_epoch(model, loader, optimizer, criterion, device)
        print(f"  Epoch {epoch+1}/3  loss={loss:.4f}")

    acc = evaluate(model, loader, device)
    print(f"  Train acc (sanity): {acc:.2%}")
    print("  [PASS] BidirectionalLSTM test complete")


def test_bert():
    """
    Test 3: BERTClassifier — forward pass shape check using random tensors.
    We do NOT download real weights here; we mock the model with
    a tiny config to stay fast and offline-friendly.

    If you want real pretrained weights, set USE_PRETRAINED = True below.
    """
    print("\n" + "=" * 60)
    print("TEST 3 — BERT Classifier")
    print("=" * 60)

    USE_PRETRAINED = False   # ← set True to download bert-base-uncased (~440 MB)

    CLASSES   = 2
    BATCH     = 4
    SEQ_LEN   = 16
    device    = torch.device("cpu")

    if USE_PRETRAINED:
        # Real pretrained BERT — needs internet + ~440 MB download
        model = BERTClassifier("bert-base-uncased", output_dim=CLASSES).to(device)
        loader = make_synthetic_transformer_data(
            n_samples=16, seq_len=SEQ_LEN, vocab_size=30522,
            n_classes=CLASSES, include_token_type=True
        )
    else:
        # Lightweight mock using a tiny BertConfig — no download needed
        from transformers import BertConfig, BertModel
        cfg = BertConfig(
            vocab_size=100,
            hidden_size=64,
            num_hidden_layers=2,
            num_attention_heads=4,
            intermediate_size=128,
        )
        # Build BERTClassifier manually with tiny config
        model = nn.Module.__new__(BERTClassifier)
        nn.Module.__init__(model)
        model.bert       = BertModel(cfg)
        model.dropout    = nn.Dropout(0.1)
        model.classifier = nn.Linear(64, CLASSES)
        model.to(device)

        loader = make_synthetic_transformer_data(
            n_samples=16, seq_len=SEQ_LEN, vocab_size=100,
            n_classes=CLASSES, include_token_type=True
        )

    # Forward pass
    batch = next(iter(loader))
    input_ids, attention_mask, token_type_ids, labels = [t.to(device) for t in batch]
    logits = model(input_ids, attention_mask, token_type_ids)
    assert logits.shape == (input_ids.size(0), CLASSES), f"Bad shape: {logits.shape}"
    print(f"  [PASS] Forward pass → logits shape {tuple(logits.shape)}")

    # 2-step training smoke test
    optimizer = optim.AdamW(model.parameters(), lr=2e-5)
    criterion = nn.CrossEntropyLoss()
    for epoch in range(2):
        loss = train_one_epoch(model, loader, optimizer, criterion, device)
        print(f"  Step {epoch+1}/2  loss={loss:.4f}")

    print("  [PASS] BERT test complete")
    print("  Tip: set USE_PRETRAINED=True for real bert-base-uncased weights")


def test_roberta():
    """
    Test 4: RoBERTaClassifier — same offline mock strategy as BERT test.
    RoBERTa uses roberta tokenizer vocab (50265) and NO token_type_ids.
    """
    print("\n" + "=" * 60)
    print("TEST 4 — RoBERTa Classifier")
    print("=" * 60)

    USE_PRETRAINED = False   # ← set True to download roberta-base (~500 MB)

    CLASSES   = 2
    BATCH     = 4
    SEQ_LEN   = 16
    device    = torch.device("cpu")

    if USE_PRETRAINED:
        model  = RoBERTaClassifier("roberta-base", output_dim=CLASSES).to(device)
        loader = make_synthetic_transformer_data(
            n_samples=16, seq_len=SEQ_LEN, vocab_size=50265,
            n_classes=CLASSES, include_token_type=False
        )
    else:
        from transformers import RobertaConfig, RobertaModel
        cfg = RobertaConfig(
            vocab_size=200,
            hidden_size=64,
            num_hidden_layers=2,
            num_attention_heads=4,
            intermediate_size=128,
        )
        model = nn.Module.__new__(RoBERTaClassifier)
        nn.Module.__init__(model)
        model.roberta    = RobertaModel(cfg)
        model.dropout    = nn.Dropout(0.1)
        model.classifier = nn.Linear(64, CLASSES)
        model.to(device)

        loader = make_synthetic_transformer_data(
            n_samples=16, seq_len=SEQ_LEN, vocab_size=200,
            n_classes=CLASSES, include_token_type=False
        )

    batch = next(iter(loader))
    input_ids, attention_mask, labels = [t.to(device) for t in batch]
    logits = model(input_ids, attention_mask)
    assert logits.shape == (input_ids.size(0), CLASSES), f"Bad shape: {logits.shape}"
    print(f"  [PASS] Forward pass → logits shape {tuple(logits.shape)}")

    optimizer = optim.AdamW(model.parameters(), lr=2e-5)
    criterion = nn.CrossEntropyLoss()
    for epoch in range(2):
        loss = train_one_epoch(model, loader, optimizer, criterion, device)
        print(f"  Step {epoch+1}/2  loss={loss:.4f}")

    print("  [PASS] RoBERTa test complete")
    print("  Tip: set USE_PRETRAINED=True for real roberta-base weights")


def test_real_world_inference():
    """
    Test 5: Real-world sentiment classification using HuggingFace pipeline.
    This test runs with a tiny distilbert model (~260 MB) if you have internet.
    If the model is not available, the test is gracefully skipped.
    """
    print("\n" + "=" * 60)
    print("TEST 5 — Real-world inference (HuggingFace pipeline)")
    print("=" * 60)

    sample_texts = [
        "I absolutely loved this movie, the acting was superb!",
        "This was the worst film I have ever seen in my life.",
        "The food was decent, nothing special but not bad either.",
    ]

    try:
        from transformers import pipeline
        # distilbert-base-uncased-finetuned-sst-2-english is small and fast
        clf = pipeline(
            "text-classification",
            model="distilbert-base-uncased-finetuned-sst-2-english",
            device=-1,   # -1 = CPU
        )
        print("  Model loaded. Running inference on sample texts...")
        for text in sample_texts:
            result = clf(text)[0]
            print(f"  [{result['label']:8s}  {result['score']:.2%}]  {text[:60]}...")
        print("  [PASS] Pipeline inference test complete")
    except Exception as exc:
        print(f"  [SKIP] Could not run pipeline test: {exc}")
        print("         Ensure you have internet access and transformers installed.")


# ─────────────────────────────────────────────────────────────────────────────
# ENTRY POINT
# ─────────────────────────────────────────────────────────────────────────────

if __name__ == "__main__":
    print("NLP Architecture Test Suite")
    print("Python / PyTorch implementation — RNN · LSTM · BERT · RoBERTa")

    test_rnn()
    test_lstm()
    test_bert()
    test_roberta()
    test_real_world_inference()

    print("\n" + "=" * 60)
    print("All tests completed.")
    print("=" * 60)

NLP Architecture Test Suite
Python / PyTorch implementation — RNN · LSTM · BERT · RoBERTa

TEST 1 — Vanilla RNN
  [PASS] Forward pass → logits shape (16, 2)
  Epoch 1/3  loss=0.7169
  Epoch 2/3  loss=0.6774
  Epoch 3/3  loss=0.6580
  Train acc (sanity): 69.50%
  [PASS] VanillaRNN test complete

TEST 2 — Bidirectional LSTM
  [PASS] Forward pass → logits shape (16, 3)
  Epoch 1/3  loss=1.0989
  Epoch 2/3  loss=1.0792
  Epoch 3/3  loss=1.0502
  Train acc (sanity): 65.33%
  [PASS] BidirectionalLSTM test complete

TEST 3 — BERT Classifier
  [PASS] Forward pass → logits shape (8, 2)
  Step 1/2  loss=0.7245
  Step 2/2  loss=0.7027
  [PASS] BERT test complete
  Tip: set USE_PRETRAINED=True for real bert-base-uncased weights

TEST 4 — RoBERTa Classifier
  [PASS] Forward pass → logits shape (8, 2)
  Step 1/2  loss=0.6895
  Step 2/2  loss=0.6810
  [PASS] RoBERTa test complete
  Tip: set USE_PRETRAINED=True for real roberta-base weights

TEST 5 — Real-world inference (HuggingFace pipeline)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

  Model loaded. Running inference on sample texts...
  [POSITIVE  99.99%]  I absolutely loved this movie, the acting was superb!...
  [NEGATIVE  99.97%]  This was the worst film I have ever seen in my life....
  [POSITIVE  98.40%]  The food was decent, nothing special but not bad either....
  [PASS] Pipeline inference test complete

All tests completed.
